# 04 - Theme mentions over time

**What this does:** reads the posts straight from `posts.parquet` (filtered by
subreddit/date below), scans each post's title and body for a curated list of
theme keywords, and counts keyword hits per theme per day.

Unlike the ticker notebooks (02/03), this does **not** rely on ticker symbols
being present. A post discussing 'HBM memory pricing' and 'Micron DRAM capacity'
scores hits for the `memory` theme even if it never uses $MU.

Produces two graphs:
1. **Raw keyword hits** - every keyword match counts as 1.
2. **Upvote-weighted hits** - each hit is weighted by the post's upvote score.

Saves `data/processed/daily_theme_counts.parquet`, which notebook 05 uses.

Edit `THEME_KEYWORDS` in `src/themes.py` to add or rename themes.

In [1]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: c:\Users\alexd\Desktop\GIC\RetailFlow1


In [2]:
# ============ TIME WINDOW - edit freely ============
# Keep this narrow to keep runs fast. Rough guide with ALL 15 subreddits:
#   1 year  ~ 700k posts  -> notebook 02 scan: a few minutes
#   all dates = 7.9M posts -> 02: ~30-60 min, 04 (themes): many HOURS
# Set both to None for the full 2008-2025 history.
START_DATE = '2021-01-01'   # inclusive,  'YYYY-MM-DD' or None
END_DATE   = '2022-01-01'   # EXCLUSIVE,  'YYYY-MM-DD' or None
# ====================================================

In [3]:
# ============ PARAMETERS - edit these ============
POSTS_PATH        = os.path.join(ROOT, 'data', 'processed', 'posts.parquet')
SUBREDDITS        = []      # ALL 15 subreddits (7.9M posts - the scan takes a while!)
THEMES_TO_PLOT    = []      # e.g. ['memory', 'ai']; [] = use the TOP_N most mentioned
TOP_N             = 6
DAILY_THEMES_OUT  = os.path.join(ROOT, 'data', 'processed', 'daily_theme_counts.parquet')
# ==================================================

In [4]:
import pandas as pd
import pyarrow.parquet as pq
from src.themes import build_daily_theme_counts, THEME_KEYWORDS

# Filtered read straight from the parquet (only needed columns).
filters = []
if SUBREDDITS:
    filters.append(('subreddit', 'in', [s.lower() for s in SUBREDDITS]))
if START_DATE:
    filters.append(('date', '>=', START_DATE))
if END_DATE:
    filters.append(('date', '<', END_DATE))

posts = pq.read_table(
    POSTS_PATH,
    columns=['date', 'title', 'selftext', 'score'],
    filters=filters if filters else None,
).to_pandas()
print('posts:', f'{len(posts):,}')
print('themes defined:', sorted(THEME_KEYWORDS.keys()))

daily = build_daily_theme_counts(posts)
daily.to_parquet(DAILY_THEMES_OUT, index=False)
print('daily theme rows:', len(daily), '-> saved', DAILY_THEMES_OUT)
daily.head()

posts: 2,832,937
themes defined: ['ai', 'ai_megacap', 'biotech_pharma', 'china_geopolitics', 'cloud_saas', 'consumer_retail', 'crypto', 'earnings', 'energy', 'ev_clean_energy', 'financials', 'gold_metals', 'ipo_spac', 'macro_rates', 'meme_stocks', 'memory', 'options_volatility', 'real_estate', 'semiconductors', 'short_squeeze']


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
daily['date'] = pd.to_datetime(daily['date'])

# Decide which themes to draw.
if THEMES_TO_PLOT:
    chosen = THEMES_TO_PLOT
else:
    totals = daily.groupby('theme')['mention_count'].sum().sort_values(ascending=False)
    chosen = list(totals.head(TOP_N).index)
print('plotting:', chosen)

def plot_signal(column, title, ylabel):
    plt.figure(figsize=(13, 6))
    for theme in chosen:
        one = daily[daily['theme'] == theme].sort_values('date')
        plt.plot(one['date'], one[column], marker='o', markersize=3, label=theme)
    plt.title(title)
    plt.xlabel('date'); plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.show()

In [ ]:
# GRAPH 1 - raw keyword hits per theme per day
plot_signal('mention_count', 'Raw theme keyword hits over time', 'keyword hits per day')

In [ ]:
# GRAPH 2 - upvote-weighted hits (each hit weighted by the post's upvotes)
plot_signal('weighted_count', 'Upvote-weighted theme mentions over time', 'sum of upvotes per day')

In [ ]:
# Summary table: total keyword hits per theme across all time
summary = (
    daily.groupby('theme')
    .agg(total_hits=('mention_count', 'sum'),
         weighted_hits=('weighted_count', 'sum'),
         days_active=('date', 'nunique'))
    .sort_values('total_hits', ascending=False)
)
summary